In [9]:
!pip install pyspark==3.5.1 
!pip install boto3 pandas python-dotenv  --upgrade


[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached boto3-1.38.33-py3-none-any.whl.metadata (6.6 kB)
  Using cached botocore-1.38.33-py3-none-any.whl.metadata (5.7 kB)
Using cached boto3-1.38.33-py3-none-any.whl (139 kB)
Using cached botocore-1.38.33-py3-none-any.whl (13.6 MB)
  Attempting uninstall: botocore
    Found existing installation: botocore 1.38.17
    Uninstalling botocore-1.38.17:
      Successfully uninstalled botocore-1.38.17
  Attempting uninstall: boto3
    Found existing installation: boto3 1.38.17
    Uninstalling boto3-1.38.17:
      Successfully uninstalled boto3-1.38.17



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

import os
import sys
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from boto3.dynamodb.conditions import Key
import pandas as pd
from pandas import json_normalize
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from dotenv import load_dotenv

# import os
from pyspark.sql import Row
# Cria a sessão do Spark
# spark = SparkSession.builder.getOrCreate()

# Força o uso do Java 11
# os.environ["JAVA_HOME"] = "C:\\Program Files\\Microsoft\\jdk-11.0.12.7-hotspot"
os.environ["JAVA_HOME"] = os.getenv('JAVA_HOME')
os.environ["PATH"] = os.environ["JAVA_HOME"] + "\\bin;" + os.environ["PATH"]

# Confirmar que o caminho foi setado
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

#Configurando variáveis de ambiente:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = os.getcwd()

#Cria uma sessão do Spark
spark = SparkSession.builder.appName("DynamoDB to PySpark").getOrCreate()


JAVA_HOME: C:\\Program Files\\Microsoft\\jdk-11.0.12.7-hotspot


In [11]:
# 1. Carrega as variáveis do arquivo .env
load_dotenv('.env')  # Ou apenas load_dotenv() se o arquivo estiver na raiz

# 2. Acessa as credenciais
aws_access = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')
region = os.getenv('AWS_DEFAULT_REGION')

# 3. Configura o cliente DynamoDB
dynamodb_console = boto3.resource(
    'dynamodb',
    aws_access_key_id=aws_access,
    aws_secret_access_key=aws_secret,
    region_name=region
)


In [12]:
dynamodb_tb = dynamodb_console.Table('pyspark-table')
user_id = '539c7a3a-d091-7074-17b5-994dde9fccd1'
response = dynamodb_tb.query(
    KeyConditionExpression=Key('PK').eq(f'USERID#{user_id}')
)

# Items é a lista de dicionário
items = response['Items']

lines = [Row(**item) for item in items]
df = spark.createDataFrame(lines)
df.printSchema()
print(response['Items'])
df.show(10)

root
 |-- data: string (nullable = true)
 |-- userId: string (nullable = true)
 |-- type_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- SK: string (nullable = true)
 |-- PK: string (nullable = true)
 |-- name: string (nullable = true)

[{'data': '2023-06-19 12:57:21', 'userId': '539c7a3a-d091-7074-17b5-994dde9fccd1', 'type_id': '2', 'status': 'done', 'SK': 'DATA#2023-06-19 12:57:21', 'PK': 'USERID#539c7a3a-d091-7074-17b5-994dde9fccd1', 'name': 'Sunt beatae'}, {'data': '2023-06-26 06:33:23', 'userId': '539c7a3a-d091-7074-17b5-994dde9fccd1', 'type_id': '1', 'status': 'done', 'SK': 'DATA#2023-06-26 06:33:23', 'PK': 'USERID#539c7a3a-d091-7074-17b5-994dde9fccd1', 'name': 'Comprar tempore'}, {'data': '2023-08-27 09:30:37', 'userId': '539c7a3a-d091-7074-17b5-994dde9fccd1', 'type_id': '2', 'status': 'todo', 'SK': 'DATA#2023-08-27 09:30:37', 'PK': 'USERID#539c7a3a-d091-7074-17b5-994dde9fccd1', 'name': 'Laboriosam dolor'}, {'data': '2023-09-12 19:56:45', 'userId': '539c7

In [13]:
abandoned_itens = df.filter(
    (col("status") == "todo") &
    (
        ((col("type_id") == "2") & (datediff(current_date(), col("data")) > 15)) |
        ((col("type_id") == "1") & (datediff(current_date(), col("data")) > 30))
    )
)

abandoned_itens.show()

+-------------------+--------------------+-------+------+--------------------+--------------------+--------------------+
|               data|              userId|type_id|status|                  SK|                  PK|                name|
+-------------------+--------------------+-------+------+--------------------+--------------------+--------------------+
|2023-08-27 09:30:37|539c7a3a-d091-707...|      2|  todo|DATA#2023-08-27 0...|USERID#539c7a3a-d...|    Laboriosam dolor|
|2023-09-12 19:56:45|539c7a3a-d091-707...|      2|  todo|DATA#2023-09-12 1...|USERID#539c7a3a-d...|Asperiores pariat...|
|2023-10-15 16:00:05|539c7a3a-d091-707...|      2|  todo|DATA#2023-10-15 1...|USERID#539c7a3a-d...|Dolore harum quos...|
|2023-12-16 05:13:08|539c7a3a-d091-707...|      2|  todo|DATA#2023-12-16 0...|USERID#539c7a3a-d...|      Ad iusto culpa|
|2024-03-10 01:25:36|539c7a3a-d091-707...|      1|  todo|DATA#2024-03-10 0...|USERID#539c7a3a-d...|  Comprar aspernatur|
|2024-03-31 16:41:51|539c7a3a-d0

In [14]:
from pyspark.sql.functions import date_format, add_months, col, to_date

data_limite = add_months(current_date(), -6)
abandoned_itens_6_months = abandoned_itens.filter(col("data") >= data_limite)

abandoned_itens_6_months.show()

+-------------------+--------------------+-------+------+--------------------+--------------------+--------------------+
|               data|              userId|type_id|status|                  SK|                  PK|                name|
+-------------------+--------------------+-------+------+--------------------+--------------------+--------------------+
|2025-02-05 16:57:46|539c7a3a-d091-707...|      2|  todo|DATA#2025-02-05 1...|USERID#539c7a3a-d...|Blanditiis corrup...|
|2025-03-12 12:43:39|539c7a3a-d091-707...|      2|  todo|DATA#2025-03-12 1...|USERID#539c7a3a-d...|                Modi|
|2025-05-09 03:37:11|539c7a3a-d091-707...|      2|  todo|DATA#2025-05-09 0...|USERID#539c7a3a-d...|Nobis fugit earum...|
+-------------------+--------------------+-------+------+--------------------+--------------------+--------------------+



In [15]:
from pyspark.sql.functions import col, to_timestamp, date_format, lit, when, current_date, add_months
from pyspark.sql import Row

# 1. Filtrar apenas os registros dos últimos 6 months
data_limite = add_months(current_date(), -6)
abandoned_6 = abandoned.filter(to_timestamp("data", "yyyy-MM-dd HH:mm:ss") >= data_limite)

# 2. Extrair o nome do mês
abandoned_6 = abandoned_6.withColumn(
    "mes_abandono",
    date_format(to_timestamp("data", "yyyy-MM-dd HH:mm:ss"), "MMMM")
)

# 3. Lista fixa com os months em inglês
months = ["January", "February", "March", "April", "May", "June"]

# 4. Função para transpor linha
def linha_transposta(df, tipo):
    contagem = df.filter(col("type_id") == tipo).groupBy("mes_abandono").count()

    if contagem.rdd.isEmpty():
        zero_data = Row(**{m: 0 for m in months})
        df_zeros = spark.createDataFrame([zero_data])
        df_zeros = df_zeros.withColumn("task_type", lit(tipo))
        return df_zeros

    for m in months:
        contagem = contagem.withColumn(m, when(col("mes_abandono") == m, col("count")).otherwise(0))

    contagem = contagem.drop("mes_abandono", "count")
    contagem = contagem.groupBy().sum()

    for m in months:
        contagem = contagem.withColumnRenamed(f"sum({m})", m)

    return contagem.withColumn("task_type", lit(tipo))

# 5. Criar linhas para cada tipo
linha_tarefas = linha_transposta(abandoned_6, "1")  # tipo_id 1 = tarefa
linha_itens = linha_transposta(abandoned_6, "2")    # tipo_id 2 = item de compra

# 6. Unir as duas linhas e garantir tipos corretos
final_df = linha_tarefas.unionByName(linha_itens)

for m in months:
    final_df = final_df.withColumn(m, col(m).cast("int"))

# 7. Renomear task_type
final_df = final_df.withColumn(
    "task_type",
    when(col("task_type") == "1", "Abandoned Task")
    .when(col("task_type") == "2", "Abandoned Shopping List")
    .otherwise(col("task_type"))
)


# 8. Reordenar colunas: task_type + months
final_df = final_df.select(["task_type"] + months)

# 9. Exibir e salvar
final_df.show()

output_dir = "relatorio_tarefas_transposto"
final_df.toPandas().to_csv("abandoned-list.csv", index=False)

# final_df.coalesce(1).write.option("header", True).mode("overwrite").csv(output_dir)


+--------------------+-------+--------+-----+-----+---+----+
|           task_type|January|February|March|April|May|June|
+--------------------+-------+--------+-----+-----+---+----+
|      Abandoned Task|      0|       0|    0|    0|  0|   0|
|Abandoned Shoppin...|      0|       1|    1|    0|  1|   0|
+--------------------+-------+--------+-----+-----+---+----+

